# SynBioCrow 2.3 — BioPKS/RetroTide Re-entry Gate
Isolated qualification only. This notebook does not run the general manuscript benchmark. It tests BioPKS in three bounded stages and checkpoints each result to Drive.


In [ ]:
import os,sys,subprocess,shutil,time,json,hashlib,zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)

REPO=Path("/content/SynBioCrow_biopks_reentry")
BIOPKS_ROOT=Path("/content/BioPKS-Pipeline")
BIOPKS_VENV=Path("/content/biopks_py310_reentry")
BIOPKS_PYTHON=BIOPKS_VENV/"bin/python"
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_reentry")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1800,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

# Fresh SynBioCrow checkout
if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

# Fresh isolated BioPKS runtime
if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
run(["git","clone","--depth","1","https://github.com/JBEI/BioPKS-Pipeline.git",str(BIOPKS_ROOT)],timeout=600)
run([sys.executable,"-m","pip","install","-q","uv"],timeout=600)
run(["uv","python","install","3.10"],timeout=900)
if BIOPKS_VENV.exists(): shutil.rmtree(BIOPKS_VENV)
run(["uv","venv","--python","3.10",str(BIOPKS_VENV)],timeout=600)

req=BIOPKS_ROOT/"requirements.txt"
patched=BIOPKS_ROOT/"requirements.synbiocrow_colab.txt"
lines=req.read_text().splitlines()
patched.write_text("\n".join(x for x in lines if x.strip().lower()!="umap==0.1.1")+"\n")

run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"-r",str(patched)],timeout=1800)
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"setuptools<81"],timeout=600)
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"--no-deps","-e",str(BIOPKS_ROOT)],timeout=900)

os.environ["SYNBIOCROW_BIOPKS_ACK_LICENSE"]="1"
os.environ["SYNBIOCROW_BIOPKS_ROOT"]=str(BIOPKS_ROOT)
os.environ["SYNBIOCROW_BIOPKS_PYTHON"]=str(BIOPKS_PYTHON)
os.environ["SYNBIOCROW_BIOPKS_RUNNER"]=str(REPO/"scripts/biopks_synbiocrow_bridge.py")

panel=REPO/"benchmarks/2.3/biopks_reentry_panel.json"
gate=REPO/"scripts/gate_biopks_reentry_2_3.py"

cmd=[
    sys.executable,str(gate),
    "--python",str(BIOPKS_PYTHON),
    "--runner",str(REPO/"scripts/biopks_synbiocrow_bridge.py"),
    "--panel",str(panel),
    "--output-dir",str(OUT),
    "--timeout-smoke","60",
    "--timeout-pks","240",
    "--timeout-combined","360",
]
print("$"," ".join(map(str,cmd)),flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout,flush=True)

report=OUT/"biopks_reentry_report.json"
assert report.is_file(),"BioPKS re-entry report was not written"
payload=json.loads(report.read_text())
print("\nBIOPKS RE-ENTRY GATE")
print(json.dumps(payload["gate"],indent=2,sort_keys=True))

bundle=OUT/"SynBioCrow_2_3_BIOPKS_REENTRY_GATE_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for fp in sorted(OUT.glob("*.json")):
        z.write(fp,fp.name)
    z.write(panel,"biopks_reentry_panel.json")
    z.write(gate,"gate_biopks_reentry_2_3.py")

print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())

if payload["gate"].get("manuscript_reentry_allowed"):
    print("RESULT: PASS_REENTRY — BioPKS may rejoin the PKS-relevant benchmark subset.")
else:
    print("RESULT:",payload["gate"].get("status"),"— keep BioPKS isolated; do not block the general benchmark.")

files.download(str(bundle))
